# Descripción de proyecto

Trabajas como analista de datos para la empresa de telecomunicaciones Megaline, la cual ofrece a sus clientes dos planes de prepago: Surf y Ultimate. El departamento comercial necesita identificar cuál de estos planes genera mayores ingresos, con el fin de optimizar la asignación del presupuesto de publicidad.

Para apoyar esta decisión, se realizará un análisis preliminar de las tarifas utilizando una muestra de clientes. El conjunto de datos incluye información de 500 usuarios de Megaline, como su ubicación, el plan que utilizan y sus patrones de consumo durante el año 2018, incluyendo la cantidad de llamadas realizadas, mensajes de texto enviados y uso de datos móviles.

Además de estimar los ingresos generados por cada plan, el análisis también se enfocará en comprender el comportamiento de los clientes, identificando patrones de uso y diferencias en el consumo entre los usuarios de cada tarifa.

El objetivo principal de este análisis es evaluar cómo se comportan los clientes según el plan contratado y determinar cuál de los planes de prepago genera mayores ingresos en promedio. Esta diferencia en ingresos será evaluada posteriormente mediante pruebas estadísticas.

# Descripción de las tarifas
Nota: Megaline redondea los segundos a minutos y los megabytes a gigabytes. Para las llamadas, cada llamada individual se redondea: incluso si la llamada duró solo un segundo, se contará como un minuto. Para el tráfico web, las sesiones web individuales no se redondean. En vez de esto, el total del mes se redondea hacia arriba. Si alguien usa 1025 megabytes este mes, se le cobrarán 2 gigabytes.

A continuación puedes ver una descripción de las tarifas:
    
Surf

Pago mensual: $20.00\
500 minutos al mes, 50 SMS y 15 GB de datos\
Si se exceden los límites del paquete:\
1 minuto: 3 centavos\
1 SMS: 3 centavos\
1 GB de datos: $10.\
Ultimate

Pago mensual: $70.\
3000 minutos al mes, 1000 SMS y 30 GB de datos.\
Si se exceden los límites del paquete:\
1 minuto: 1 centavo\
1 SMS: 1 centavo\
1 GB de datos: $7.00

### Análisis general de los datos
1- Obtener todas las bases de datos y librerías necesarias

2- Visualizar el tipo de datos que se van a manejar

3- Visualizar posibles duplicados 

4- Corregir datos

5- Enriquecer los datos

In [18]:
import pandas as pd
from scipy import stats as st
import numpy as np
import matplotlib.pyplot as plt
import math
# -----------------------------------------------------------------------#
meg_calls = pd.read_csv('datasets/megaline_calls.csv')
meg_internet = pd.read_csv('datasets/megaline_internet.csv')
meg_msm = pd.read_csv('datasets/megaline_messages.csv')
meg_plans = pd.read_csv('datasets/megaline_plans.csv')
meg_users = pd.read_csv('datasets/megaline_users.csv')

In [19]:
meg_calls.info()
meg_calls.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137735 entries, 0 to 137734
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   id         137735 non-null  object 
 1   user_id    137735 non-null  int64  
 2   call_date  137735 non-null  object 
 3   duration   137735 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 4.2+ MB


,id,user_id,call_date,duration
20494,1077_461,1077,2018-03-02,0.00
74025,1263_9,1263,2018-10-06,0.00
76340,1270_434,1270,2018-09-26,3.34
6713,1031_504,1031,2018-11-07,0.00
52237,1187_430,1187,2018-09-05,7.38
72898,1257_638,1257,2018-08-12,6.86
40612,1150_144,1150,2018-10-13,0.00
97734,1348_652,1348,2018-11-26,17.76
105643,1373_123,1373,2018-10-23,6.20
47079,1171_110,1171,2018-12-01,1.94


In [20]:
meg_internet.info()
meg_internet.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104825 entries, 0 to 104824
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   id            104825 non-null  object 
 1   user_id       104825 non-null  int64  
 2   session_date  104825 non-null  object 
 3   mb_used       104825 non-null  float64
dtypes: float64(1), int64(1), object(2)
memory usage: 3.2+ MB


,id,user_id,session_date,mb_used
49556,1222_55,1222,2018-12-14,0.00
5172,1028_755,1028,2018-05-29,253.09
65441,1301_131,1301,2018-12-16,56.77
68729,1321_85,1321,2018-07-27,638.03
100388,1475_181,1475,2018-09-12,0.00
79541,1371_109,1371,2018-12-09,0.00
18716,1083_385,1083,2018-11-24,584.21
79272,1369_13,1369,2018-11-28,294.40
39666,1178_449,1178,2018-11-29,0.00
5793,1031_471,1031,2018-11-04,420.03


In [21]:
meg_msm.info()
meg_msm.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76051 entries, 0 to 76050
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id            76051 non-null  object
 1   user_id       76051 non-null  int64 
 2   message_date  76051 non-null  object
dtypes: int64(1), object(2)
memory usage: 1.7+ MB


,id,user_id,message_date
26848,1164_98,1164,2018-09-06
43618,1289_196,1289,2018-12-14
58644,1373_62,1373,2018-12-25
5970,1055_88,1055,2018-12-19
9712,1072_151,1072,2018-09-06
69584,1453_32,1453,2018-12-07
42161,1272_56,1272,2018-12-16
25945,1155_338,1155,2018-03-26
29227,1179_23,1179,2018-08-07
3290,1036_125,1036,2018-07-30


In [22]:
meg_plans.info()
meg_plans.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   messages_included      2 non-null      int64  
 1   mb_per_month_included  2 non-null      int64  
 2   minutes_included       2 non-null      int64  
 3   usd_monthly_pay        2 non-null      int64  
 4   usd_per_gb             2 non-null      int64  
 5   usd_per_message        2 non-null      float64
 6   usd_per_minute         2 non-null      float64
 7   plan_name              2 non-null      object 
dtypes: float64(2), int64(5), object(1)
memory usage: 260.0+ bytes


,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,plan_name
0,50,15360,500,20,10,0.03,0.03,surf
1,1000,30720,3000,70,7,0.01,0.01,ultimate


In [23]:
meg_users.info()
meg_users.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   user_id     500 non-null    int64 
 1   first_name  500 non-null    object
 2   last_name   500 non-null    object
 3   age         500 non-null    int64 
 4   city        500 non-null    object
 5   reg_date    500 non-null    object
 6   plan        500 non-null    object
 7   churn_date  34 non-null     object
dtypes: int64(2), object(6)
memory usage: 31.4+ KB


,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
405,1405,Shella,Hunter,34,"Indianapolis-Carmel-Anderson, IN MSA",2018-06-22,ultimate,NaN
216,1216,Reita,Atkins,29,"San Francisco-Oakland-Berkeley, CA MSA",2018-06-19,surf,NaN
164,1164,Kam,Macias,31,"Chicago-Naperville-Elgin, IL-IN-WI MSA",2018-02-17,ultimate,NaN
321,1321,Karlyn,Jimenez,19,"North Port-Sarasota-Bradenton, FL MSA",2018-05-31,surf,NaN
327,1327,Natosha,Peck,20,"New York-Newark-Jersey City, NY-NJ-PA MSA",2018-02-17,surf,NaN


Todos estos datasets tienen distintos propositos, como los 3 primeros que tiene informacion acerca del consumo de cada servicio proporcionado, no para internet, otro para mensajes y etc.
Uno con infomacion acerca de los usuarios y otro acerca de los datos de las tarifas que ofrece esta empresa.

Se puede ver que en todos los datasets el tipo de dato para las fechas es incorrecto.

En "churn date" hay información vacía, sin embargo en este caso se puede conservar este estado ya que indica que no se han cancelado las suscripciones.

En la base de datos meg_plans, los nombres de las columnas son demasiado largas, esto se puede reducir.

aqui se vio algo interesante y es que en varias ocaciones aparece el comsumo de internet en 0 o llamdas con duracion de 0, esto puede significar dos cosas que se estan tomando todos los dias sin excepcion o que se recopilo esta informacion por error

Además, en base a las condiciones de la suscripción se debe de redondear hacia arriba el internet consumido, así que esto se debe de corregir.

# Comprobacion de informacion

In [ ]:
comprobacion = meg_internet.sort_values(
    by=['user_id', 'session_date'], ascending=True)
usuario = np.random.choice(meg_users['user_id'])
print(comprobacion[(comprobacion)['user_id'] == usuario])

            id  user_id session_date  mb_used
6179  1034_163     1034   2018-08-14   791.50
6180  1034_238     1034   2018-08-14   205.92


Aqui lo que se hizo es agarrar a un usuario al alzar y mostrar todo su historial de consumo, y demostro que incluso en un mismo dia hay registros con 0 de consumo lo cual se puede explicar si un usuario inicio sesion pero no descargo nada... pasara lo mismo con las llamadas?

In [ ]:
comprobacion = meg_calls.sort_values(
    by=['user_id', 'call_date'], ascending=True)
usuario = np.random.choice(meg_users['user_id'])
print(comprobacion[(comprobacion)['user_id'] == usuario])

            id  user_id   call_date  duration
7333  1034_194     1034  2018-08-14      0.16
7332  1034_129     1034  2018-08-15      5.15


Demuestra el mismo comportamiento teniendo como explicacion que el conteo de llamdas se cuenta desde el momento que contesta la otra persona, los registros en 0 puede significar que esta llamada no fue contestada.

Dado que nuestro objetivo de investigacion se centra principalmente en analizar el consumo realizadio y las ganancias generadas de cada plan esta informacion no es relevante, no se eliminara pero tampoco sera tomada en cuenta

asi que continuaremos a la correcion de los datos de los datasets presentados

In [26]:
# Cambio al tipo fecha de las columnas marcadas#

meg_users[['churn_date', 'reg_date']] = meg_users[[
    'churn_date', 'reg_date']].apply(pd.to_datetime)
meg_calls['call_date'] = pd.to_datetime(meg_calls['call_date'])
meg_internet['session_date'] = pd.to_datetime(meg_internet['session_date'])
meg_msm['message_date'] = pd.to_datetime(meg_msm['message_date'])


# Eliminar los datos duplicados#

meg_internet.drop_duplicates(inplace=True)
meg_calls.drop_duplicates(inplace=True)
meg_users.drop_duplicates(inplace=True)
meg_msm.drop_duplicates(inplace=True)

# Renombrar algunas columnas#

meg_plans = meg_plans.rename(columns={'plan_name': 'plan'})

In [27]:
meg_calls.info()
meg_calls.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137735 entries, 0 to 137734
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   id         137735 non-null  object        
 1   user_id    137735 non-null  int64         
 2   call_date  137735 non-null  datetime64[ns]
 3   duration   137735 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 4.2+ MB


,id,user_id,call_date,duration
48063,1174_40,1174,2018-07-26,0.00
87363,1320_1032,1320,2018-10-17,0.00
107998,1382_1002,1382,2018-09-13,11.13
50569,1182_160,1182,2018-07-28,12.26
44439,1160_455,1160,2018-09-15,0.00
43637,1159_233,1159,2018-10-07,0.00
75071,1264_442,1264,2018-09-28,8.71
53770,1193_210,1193,2018-09-11,0.00
76975,1277_51,1277,2018-06-16,13.57
39251,1146_453,1146,2018-11-14,2.22


In [28]:
meg_internet.info()
meg_internet.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104825 entries, 0 to 104824
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype         
---  ------        --------------   -----         
 0   id            104825 non-null  object        
 1   user_id       104825 non-null  int64         
 2   session_date  104825 non-null  datetime64[ns]
 3   mb_used       104825 non-null  float64       
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 3.2+ MB


,id,user_id,session_date,mb_used
70616,1329_356,1329,2018-10-30,592.24
98787,1466_268,1466,2018-11-13,245.30
99374,1468_106,1468,2018-11-16,413.28
67825,1316_39,1316,2018-12-07,451.42
4220,1027_48,1027,2018-11-27,443.66
58612,1263_416,1263,2018-11-27,369.77
65602,1302_148,1302,2018-09-28,364.27
5737,1031_375,1031,2018-07-30,200.78
53054,1240_26,1240,2018-09-25,357.96
93691,1437_278,1437,2018-11-17,0.00


In [29]:
meg_msm.info()
meg_msm.sample(20)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 76051 entries, 0 to 76050
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   id            76051 non-null  object        
 1   user_id       76051 non-null  int64         
 2   message_date  76051 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 1.7+ MB


,id,user_id,message_date
70434,1458_108,1458,2018-10-17
17365,1114_231,1114,2018-10-27
73480,1472_3,1472,2018-07-15
35607,1232_5,1232,2018-10-10
17196,1113_319,1113,2018-12-04
23692,1144_160,1144,2018-07-15
74486,1478_14,1478,2018-09-07
58549,1372_21,1372,2018-12-18
67481,1438_47,1438,2018-11-27
27792,1172_249,1172,2018-11-21


In [30]:
meg_plans.info()
meg_plans.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 8 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   messages_included      2 non-null      int64  
 1   mb_per_month_included  2 non-null      int64  
 2   minutes_included       2 non-null      int64  
 3   usd_monthly_pay        2 non-null      int64  
 4   usd_per_gb             2 non-null      int64  
 5   usd_per_message        2 non-null      float64
 6   usd_per_minute         2 non-null      float64
 7   plan                   2 non-null      object 
dtypes: float64(2), int64(5), object(1)
memory usage: 260.0+ bytes


,messages_included,mb_per_month_included,minutes_included,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,plan
0,50,15360,500,20,10,0.03,0.03,surf
1,1000,30720,3000,70,7,0.01,0.01,ultimate


In [31]:
meg_users.info()
meg_users.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     500 non-null    int64         
 1   first_name  500 non-null    object        
 2   last_name   500 non-null    object        
 3   age         500 non-null    int64         
 4   city        500 non-null    object        
 5   reg_date    500 non-null    datetime64[ns]
 6   plan        500 non-null    object        
 7   churn_date  34 non-null     datetime64[ns]
dtypes: datetime64[ns](2), int64(2), object(4)
memory usage: 31.4+ KB


,user_id,first_name,last_name,age,city,reg_date,plan,churn_date
224,1224,Kelly,Cole,74,"Atlanta-Sandy Springs-Roswell, GA MSA",2018-06-28,ultimate,NaT
235,1235,Felton,Nguyen,50,"Minneapolis-St. Paul-Bloomington, MN-WI MSA",2018-02-20,surf,NaT
236,1236,Odell,Juarez,74,"Minneapolis-St. Paul-Bloomington, MN-WI MSA",2018-04-04,ultimate,NaT
59,1059,Avril,Richardson,34,"Chicago-Naperville-Elgin, IL-IN-WI MSA",2018-04-22,ultimate,NaT
283,1283,Alan,Burgess,38,"Charleston-North Charleston, SC MSA",2018-06-16,ultimate,NaT


# Enriquecimiento de datos

Se extrajo el componente mensual de las variables de fecha en los distintos datasets para estandarizar la dimensión temporal del análisis y permitir la agregación consistente de la actividad de los usuarios por ciclo de facturación. Adicionalmente, se convirtió el límite de datos incluidos de megabytes a gigabytes utilizando redondeo hacia arriba (ceil) para reflejar las reglas de facturación del negocio y asegurar la precisión en el cálculo de ingresos y excedentes de consumo.

In [ ]:
meg_msm["period"] = meg_msm["message_date"].dt.month
meg_calls["period"] = meg_calls["call_date"].dt.month
meg_users['month_reg'] = meg_users['reg_date'].dt.month
meg_users['month_churn'] = meg_users['churn_date'].dt.month
meg_internet["period"] = meg_internet["session_date"].dt.month
meg_plans['mb_per_month_included'] = np.ceil(
    meg_plans['mb_per_month_included'] / 1024)

A continuacion haremos un nuevo dataset para poder calcular las métricas a nivel de usuario y período mensual, con esto para poder empezar a analizar los comportamientos de los clientes dentro de cada ciclo de facturación.

In [ ]:
# Agregar los datos por usuario y por periodo de cada dataset#
llamadas_mes = meg_calls.groupby(['user_id', 'period']).size().reset_index()
min_llamadas = meg_calls.groupby(['user_id', 'period'])[
    'duration'].sum().reset_index()
n_mensajes = meg_msm.groupby(['user_id', 'period']).size().reset_index()
inter_trafic = meg_internet.groupby(['user_id', 'period'])[
    'mb_used'].sum().reset_index()

# Renombre de columnas#

llamadas_mes = llamadas_mes.rename(columns={0: 'calls_n'})
n_mensajes.rename(columns={0: 'n_msm'}, inplace=True)

Después de obtener todos los datos de manera individual los agrupamos todos para poder leer la información de una manera más limpia y organizada.
Además, aquí se redondean dos datos: MB usados y duración de llamadas hacia arriba.

In [ ]:
total = pd.merge(llamadas_mes, min_llamadas, on=[
                 'user_id', 'period'], how='outer')
total = pd.merge(total, inter_trafic, on=['user_id', 'period'], how='outer')
total = pd.merge(total, n_mensajes, on=['user_id', 'period'], how='outer')
total = pd.merge(
    total, meg_users[['user_id', 'plan', 'city']], on='user_id', how='outer')
total = total.fillna(0)
total['mb_used'] = np.ceil(total['mb_used'] / 1024)
total['duration'] = np.ceil(total['duration'])
total.info()
total
total.to_excel('datasets/total.xlsx', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2303 entries, 0 to 2302
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   user_id   2303 non-null   int64  
 1   period    2303 non-null   float64
 2   calls_n   2303 non-null   float64
 3   duration  2303 non-null   float64
 4   mb_used   2303 non-null   float64
 5   n_msm     2303 non-null   float64
 6   plan      2303 non-null   object 
 7   city      2303 non-null   object 
dtypes: float64(5), int64(1), object(2)
memory usage: 144.1+ KB


Con el dataset anterior se hará un nuevo dataset en donde se expondrán dos cosas: los límites que uno puede consumir por período y, al lado, lo que ha consumido extra en ese período. Teniendo como columna final lo que el usuario va a pagar en total, este total se calculará teniendo en cuenta la tarifa mencionada al inicio del proyecto y sumando lo extra por cada plan.

In [ ]:
# Calculamos los excedentes
total_extra = total.merge(meg_plans, on='plan')
pares = [
    ('duration', 'minutes_included', 'minutos'),
    ('n_msm', 'messages_included', 'mensajes'),
    ('mb_used', 'mb_per_month_included', 'mb')
]
for usado, limite, prefijo in pares:
    total_extra[f'{prefijo}_dentro_plan'] = total_extra[[
        usado, limite]].min(axis=1)
    total_extra[f'{prefijo}_excedido'] = (
        total_extra[usado] - total_extra[limite]).clip(lower=0)
nuevo_orden = [
    'user_id', 'period', 'plan',
    'messages_included', 'mensajes_dentro_plan', 'mensajes_excedido',
    'mb_per_month_included', 'mb_dentro_plan', 'mb_excedido',
    'minutes_included', 'minutos_dentro_plan', 'minutos_excedido',
    'usd_monthly_pay', 'usd_per_gb', 'usd_per_message', 'usd_per_minute'
]
total_extra = total_extra[nuevo_orden]
total_extra['pago_total'] = (total_extra['usd_monthly_pay'] +
                             (total_extra['mb_excedido'] * total_extra['usd_per_gb']) +
                             (total_extra['mensajes_excedido'] * total_extra['usd_per_message']) +
                             (total_extra['minutos_excedido'] * total_extra['usd_per_minute']))
total_extra.info()
total_extra

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2303 entries, 0 to 2302
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   user_id                2303 non-null   int64  
 1   period                 2303 non-null   float64
 2   plan                   2303 non-null   object 
 3   messages_included      2303 non-null   int64  
 4   mensajes_dentro_plan   2303 non-null   float64
 5   mensajes_excedido      2303 non-null   float64
 6   mb_per_month_included  2303 non-null   float64
 7   mb_dentro_plan         2303 non-null   float64
 8   mb_excedido            2303 non-null   float64
 9   minutes_included       2303 non-null   int64  
 10  minutos_dentro_plan    2303 non-null   float64
 11  minutos_excedido       2303 non-null   float64
 12  usd_monthly_pay        2303 non-null   int64  
 13  usd_per_gb             2303 non-null   int64  
 14  usd_per_message        2303 non-null   float64
 15  usd_

,user_id,period,plan,messages_included,mensajes_dentro_plan,mensajes_excedido,mb_per_month_included,mb_dentro_plan,mb_excedido,minutes_included,minutos_dentro_plan,minutos_excedido,usd_monthly_pay,usd_per_gb,usd_per_message,usd_per_minute,pago_total
0,1000,12.0,ultimate,1000,11.0,0.0,30.0,2.0,0.0,3000,117.0,0.0,70,7,0.01,0.01,70.00
1,1001,8.0,surf,50,30.0,0.0,15.0,7.0,0.0,500,172.0,0.0,20,10,0.03,0.03,20.00
2,1001,9.0,surf,50,44.0,0.0,15.0,14.0,0.0,500,298.0,0.0,20,10,0.03,0.03,20.00
3,1001,10.0,surf,50,50.0,3.0,15.0,15.0,7.0,500,375.0,0.0,20,10,0.03,0.03,90.09
4,1001,11.0,surf,50,36.0,0.0,15.0,15.0,4.0,500,405.0,0.0,20,10,0.03,0.03,60.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2298,1498,12.0,surf,50,0.0,0.0,15.0,15.0,8.0,500,325.0,0.0,20,10,0.03,0.03,100.00
2299,1499,9.0,surf,50,0.0,0.0,15.0,13.0,0.0,500,331.0,0.0,20,10,0.03,0.03,20.00
2300,1499,10.0,surf,50,0.0,0.0,15.0,15.0,5.0,500,364.0,0.0,20,10,0.03,0.03,70.00
2301,1499,11.0,surf,50,0.0,0.0,15.0,15.0,2.0,500,289.0,0.0,20,10,0.03,0.03,40.00


In [ ]:
# calculamos los costos
def calcular_ingresos(df):
    # Calcular cargos por excedentes
    costo_minutos = df['minutos_excedido'] * df['usd_per_minute']
    costo_mensajes = df['mensajes_excedido'] * df['usd_per_message']
    costo_datos = df['mb_excedido'] * df['usd_per_gb']
    df['ingreso_mensual'] = df['usd_monthly_pay'] +\
        costo_minutos + costo_mensajes + costo_datos

    return df


df_s = calcular_ingresos(total_extra)
df_s
total = pd.merge(
    total,
    df_s[['user_id', 'period', 'ingreso_mensual']],
    on=['user_id', 'period'],
    how='outer'
)
total.info()
total

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2303 entries, 0 to 2302
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   user_id          2303 non-null   int64  
 1   period           2303 non-null   float64
 2   calls_n          2303 non-null   float64
 3   duration         2303 non-null   float64
 4   mb_used          2303 non-null   float64
 5   n_msm            2303 non-null   float64
 6   plan             2303 non-null   object 
 7   city             2303 non-null   object 
 8   ingreso_mensual  2303 non-null   float64
dtypes: float64(6), int64(1), object(2)
memory usage: 162.1+ KB


,user_id,period,calls_n,duration,mb_used,n_msm,plan,city,ingreso_mensual
0,1000,12.0,16.0,117.0,2.0,11.0,ultimate,"Atlanta-Sandy Springs-Roswell, GA MSA",70.00
1,1001,8.0,22.0,172.0,7.0,30.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",20.00
2,1001,9.0,38.0,298.0,14.0,44.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",20.00
3,1001,10.0,47.0,375.0,22.0,53.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",90.09
4,1001,11.0,49.0,405.0,19.0,36.0,surf,"Seattle-Tacoma-Bellevue, WA MSA",60.00
...,...,...,...,...,...,...,...,...,...
2298,1498,12.0,32.0,325.0,23.0,0.0,surf,"New York-Newark-Jersey City, NY-NJ-PA MSA",100.00
2299,1499,9.0,35.0,331.0,13.0,0.0,surf,"Orlando-Kissimmee-Sanford, FL MSA",20.00
2300,1499,10.0,41.0,364.0,20.0,0.0,surf,"Orlando-Kissimmee-Sanford, FL MSA",70.00
2301,1499,11.0,39.0,289.0,17.0,0.0,surf,"Orlando-Kissimmee-Sanford, FL MSA",40.00
